In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import polars as pl
import numpy as np
import torch
import zarr
from tqdm import tqdm

from anngeno import AnnGeno
from scripts import get_burdens, get_correlations

import multiprocessing

device = "cuda" if torch.cuda.is_available() else "cpu"
num_cores = multiprocessing.cpu_count()
print(device, num_cores)

## Read Anngeno file

In [ ]:
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
ag = AnnGeno(anngeno_path,  filemode="r", low_mem=True)
ag

## Read gene-trait associations

In [ ]:
gt = pl.read_parquet("/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq")[['gene_id', 'gene_symbol', 'description']].unique()

gt = gt.with_columns(pl.col("description").str.to_lowercase().alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all("-", "").alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all(" ", "_").alias("phenotype")).drop(['description'])
gt

### Subset associations to genes and phenoptypes available in the small Anngeno

In [ ]:
subsest_gt = gt.filter(
    pl.col("gene_id").is_in(ag.annotations.select(pl.col("region")).collect().unique()['region'])
    ).filter(
    pl.col("phenotype").is_in(ag.phenotypes.columns)
    )

subsest_gt

### Small subset of associations to sanity check

In [ ]:
subsest_gt.filter(pl.col('gene_symbol').is_in(["CETP", "LCAT"])).write_parquet("/home/dnanexus/subset_genes.pq")

## Compute burdens for the small subset

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

all_annotation_list

In [ ]:
%%time

anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
associations_df_path = '/home/dnanexus/subset_genes.pq'

gene_burdens_sum_df, gene_burdens_max_df, gene_burdens_top2_df, sample_id_arr, gene_id_list = get_burdens.get_burdens_array(
    anngeno_path=anngeno_path,
    associations_df_path=associations_df_path,
    maf=0.001,
    annotation_list=all_annotation_list,
    max_burden=True,
    only_snps=False,
    n_jobs=1,
    batch_size=2,
    device=device,
)


In [ ]:
gene_burdens_max_df.shape

In [ ]:
a = torch.tensor(gene_burdens_max_df[:,0,:], device=device).transpose(1,0)
a.shape

## Phenotype GIS plot

In [ ]:
all_annotation_list

In [ ]:
gene_id_list

In [ ]:
annotation = 'promoterAI' #'am_pathogenicity'
gene_id = 'ENSG00000213398' #'ENSG00000130164' # LDLR

# Get indices for the gene and annotation
gene_idx = gene_id_list.index(gene_id)
annotation_idx = all_annotation_list.index(annotation)

# Get the burden array for the specific gene and annotation
gis_df = pl.DataFrame({
    "sample": sample_id_arr,
    "gis_sum": gene_burdens_sum_df[:, gene_idx, annotation_idx],
    "gis_max": gene_burdens_max_df[:, gene_idx, annotation_idx],
    "gis_top2": gene_burdens_top2_df[:, gene_idx, annotation_idx]
})
gis_df

In [ ]:
trait = "hdl_cholesterol"
# pheno_df = pl.read_parquet("/home/dnanexus/data_dir/phenotypes_corr/ldl_direct_prs_corrected.parquet")
pheno_df = pl.read_parquet(f"/home/dnanexus/data_dir/phenotypes_corr/{trait}_prs_corrected.parquet")
plt_df = gis_df.join(pheno_df, on="sample", how="inner")
plt_df

In [ ]:
from plotnine import *

(
    ggplot(plt_df, aes(x='gis_max', y=f'{trait}_prs_corrected')) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=False) +
    labs(
        y=f'{trait} (prs corrected)'
        ) +
    theme_bw()
)

## Create and save Zarr

In [ ]:
gt = pl.read_parquet("/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq").filter(pl.col('annotation').str.contains('pLoF'))[['gene_id', 'gene_symbol', 'description']].unique()

gt = gt.with_columns(pl.col("description").str.to_lowercase().alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all("-", "").alias("description"))
gt = gt.with_columns(pl.col("description").str.replace_all(" ", "_").alias("phenotype")).drop(['description'])

# PRS corrected phenos available in the ag.phenotypes
ag_phenos = ['apolipoprotein_a', 'apolipoprotein_b', 'cholesterol', 'hdl_cholesterol', 'ldl_direct', 'standing_height', 'triglycerides']

subsest_gt = gt.filter(
    pl.col("gene_id").is_in(ag.annotations.select(pl.col("region")).collect().unique()['region'])
    ).filter(
    pl.col("phenotype").is_in(ag_phenos)
    )

subsest_gt.write_parquet("/home/dnanexus/subset_genes.pq")
subsest_gt

In [ ]:
%%time

config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
anngeno_path = "/home/dnanexus/data_dir/anngeno_training.ag"
associations_df_path = '/home/dnanexus/subset_genes.pq'
output_zarr = "/home/dnanexus/250624_small_anngeno_all_annotations_burdens.zarr"
sample_set = set(pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv').select(pl.col('eid').cast(pl.Utf8))['eid'])

get_burdens.compute_and_store_burdens(
    config_path=config_path,
    associations_df_path=associations_df_path,
    output_zarr=output_zarr,
    sample_set=sample_set,
    gene_chunk_size=10,
    sample_chunk_size=25_000,
    device=device,
)

### Read zarr

In [ ]:
zarr_burdens_path = '/home/dnanexus/250624_small_anngeno_all_annotations_burdens.zarr'
zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
sample_list = zarr_group["samples"][:]
gene_list = zarr_group["genes"][:]
annotation_list = zarr_group["annotations"][:]

In [ ]:
zarr_group["top2_burdens"][:, :, :].shape

In [ ]:
zarr_group["top2_burdens"][:, :, :]

## Save all correlations